# Day 3 - Lab 1: How models learn (regression and classification)

**Goal:** see what "learning" actually is by coding gradient descent by hand, then use scikit-learn to fit real regression and classification models on the feature table you built on Day 2, and evaluate them honestly, meeting the leakage trap head on.

A model is just a set of numbers (weights) chosen to make its predictions match the data. Everything below is that idea, made concrete.

## 1. Load the model-ready table from Day 2

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()
df = pd.read_csv(DATA / 'processed' / 'service_requests_features.csv')
print('shape:', df.shape)
print('rows with a known sla_met:', int(df['sla_met'].notna().sum()))
df.head(3)

## 2. What is actually known when a request is logged?

This is the single most important idea in the day. Two columns in the table are **outcomes**, not inputs:

- `resolution_hours` is how long the request eventually took. You do not know it at logging time.
- `satisfaction_score` is collected *after* the request is resolved.

If you let a model use an outcome to predict another outcome, it will look brilliant in training and be useless in practice. That is **data leakage**. So we define an honest feature set: only things known the moment a request arrives.

In [ ]:
honest = ['submitted_hour', 'submitted_dow', 'submitted_month', 'is_weekend',
          'is_digital', 'population', 'priority_rank', 'target_resolution_hours']

lab = df[df['sla_met'].notna()].copy()
lab['sla_met'] = lab['sla_met'].astype(int)
print('honest features:', len(honest))
print('labelled rows:', len(lab))

## 3. Linear regression from scratch: watch a model learn

We will predict `resolution_hours` from the honest features. The learning rule is simple: start with weights of zero, look at how wrong you are, and nudge every weight a little in the direction that reduces the error. Repeat. That nudging is **gradient descent**.

Run the cell and watch the loss (mean squared error) fall.

In [ ]:
from sklearn.preprocessing import StandardScaler

reg = df[df['resolution_hours'].notna() & df['resolution_hours'].between(0, 8760)].copy()
Xr = StandardScaler().fit_transform(reg[honest].values)
yr = reg['resolution_hours'].values
yr_c = (yr - yr.mean()) / yr.std()          # centre + scale the target for stable steps

n, d = Xr.shape
w = np.zeros(d); b = 0.0; lr = 0.1
checkpoints = [1, 20, 50, 100, 200, 400]
history = []
for it in range(1, 401):
    pred = Xr @ w + b
    err = pred - yr_c
    history.append(np.mean(err**2))             # record the loss at every step
    if it in checkpoints:                       # print BEFORE the step, so iter 1 is the starting loss
        print(f'iter {it:3d}  MSE {np.mean(err**2):.4f}')
    w -= lr * (Xr.T @ err) / n
    b -= lr * err.mean()
print('\nlearned weights (scaled):', dict(zip(honest, np.round(w, 3))))

### See it: the model learning

The printed numbers are the same story, but the curve is what makes it stick. This is *the* shape of machine learning: a steep early drop as the model finds the obvious signal, then a flattening as it runs out of things to learn.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history, linewidth=2)
axes[0].set_title('Linear regression: loss falling with each step')
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('mean squared error')
axes[0].grid(alpha=0.3)

axes[1].plot(history[:60], linewidth=2, color='darkorange')
axes[1].set_title('The first 60 steps, zoomed')
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('mean squared error')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

That is the whole idea of supervised learning: a loss to measure wrongness, and small repeated steps downhill. scikit-learn does exactly this, faster and with a closed-form solution for linear regression.

## 4. The same model in scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Xtr, Xte, ytr, yte = train_test_split(reg[honest], yr, test_size=0.25, random_state=42)

# TODO: fit a LinearRegression on (Xtr, ytr), predict on Xte, and print MAE, RMSE and R2.
# Expected: MAE about 26 hours, R2 about 0.61.


An R2 around 0.6 and a mean error near a day. Useful, not magic: the honest features carry real signal about how long a request will take, but not the whole story.

## 5. Logistic regression from scratch

Classification reuses the *same* machinery with two changes: squeeze the output through a **sigmoid** so it lands between 0 and 1 (a probability), and measure wrongness with **log-loss** instead of squared error. The gradient ends up looking almost identical.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

Xc = StandardScaler().fit_transform(lab[honest].values)
yc = lab['sla_met'].values.astype(float)

n, d = Xc.shape
w = np.zeros(d); b = 0.0; lr = 0.3
checkpoints = [1, 20, 50, 100, 200, 400]
for it in range(1, 401):
    p = sigmoid(Xc @ w + b)
    err = p - yc                     # same shape of update as linear regression
    if it in checkpoints:
        loss = -np.mean(yc*np.log(p+1e-9) + (1-yc)*np.log(1-p+1e-9))
        print(f'iter {it:3d}  log-loss {loss:.4f}')
    w -= lr * (Xc.T @ err) / n
    b -= lr * err.mean()

## 6. Logistic regression in scikit-learn, and the leakage trap

Now we fit the real classifier for `sla_met` (was the SLA met). First the wrong way, letting `resolution_hours` in, then the honest way.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

y = lab['sla_met']
def fit_report(feats, label):
    X = lab[feats]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
    p = m.predict(Xte); pr = m.predict_proba(Xte)[:, 1]
    print(f'{label:32s} accuracy {accuracy_score(yte, p):.3f}   AUC {roc_auc_score(yte, pr):.3f}')
    return m

# TODO: call fit_report twice.
#   1) with the honest features PLUS 'resolution_hours'  -> the leaked model
#   2) with the honest features only                     -> the honest model
# Keep the honest model in a variable called `model`; you need it in the next step.


The leaked model scores around 99% because `resolution_hours` *defines* whether the SLA was met, so the model is copying the answer. The honest model drops to about 69%. **That fall is the lesson.** A result that looks too good almost always means a feature is leaking the outcome.

## 7. Read the result: the confusion matrix

Accuracy hides the errors that matter. Frame the positive class as an **SLA miss**, the thing a supervisor would want flagged early, and split the predictions four ways.

In [ ]:
from sklearn.metrics import confusion_matrix

Xtr, Xte, ytr, yte = train_test_split(lab[honest], y, test_size=0.25, random_state=42, stratify=y)
pred = model.predict(Xte)

# TODO: frame the positive class as an SLA MISS (sla_met == 0), build the confusion
# matrix, and print TP, FP, FN, TN plus precision and recall for the miss class.
# Hint: confusion_matrix(miss_true, miss_pred).ravel() returns tn, fp, fn, tp.
# IMPORTANT: name your four counts exactly `tn`, `fp`, `fn`, `tp` -- the chart in the
# next cell uses them.


Each cell has a cost. A **false negative** is a breach nobody was warned about. A **false positive** is a supervisor's time spent on a request that was fine. Here they are not equally painful, which is why accuracy alone is the wrong score: you would tune this model toward **recall**, accepting more false alarms to miss fewer real breaches.

### See it: the confusion matrix as a picture

The grid below is the same four numbers, but laid out the way you should think about them. The green diagonal is what the model got right; the red off-diagonal is the cost.

In [ ]:
import matplotlib.pyplot as plt

cm = np.array([[tn, fp], [fn, tp]])
names = np.array([['TN\ncorrectly left alone', 'FP\nfalse alarm'],
                  ['FN\nmissed breach', 'TP\ncaught the miss']])
colours = np.array([[0.85, 0.35], [0.15, 0.85]])   # green-ness: right vs wrong

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(colours, cmap='RdYlGn', vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{names[i, j]}\n\n{cm[i, j]}', ha='center', va='center',
                fontsize=11, fontweight='bold')
ax.set_xticks([0, 1]); ax.set_xticklabels(['predicted: OK', 'predicted: MISS'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['actual: OK', 'actual: MISS'])
ax.set_title('Positive class = the request MISSED its SLA', pad=12)
plt.tight_layout(); plt.show()

## 8. Would you switch it on?

The honest model catches about three quarters of real breaches (recall near 0.76) at roughly 0.63 precision, so for every four genuine breaches it flags three, at the cost of some false alarms. That is the actual question at the end of every model: not "is the accuracy high" but "is this good enough to change what someone does on Monday". Hold that judgement; in Lab 2 you will test whether more powerful models actually do any better here.

## Your turn

1. The classifier uses a default decision threshold of 0.5 on the predicted probability. Lower it (predict a miss when probability of *meeting* the SLA is below, say, 0.6). What happens to precision and recall for the miss class, and why?
2. Add `satisfaction_score` back into the honest feature list and refit. Accuracy improves. Explain in one sentence why using it would still be cheating in a real deployment.
3. Look at the linear regression coefficients from section 4 (`lin.coef_`). Which feature moves the predicted resolution time the most, and does its sign match your intuition?

In [ ]:
# your turn
